## Postional Encoding

In [ ]:
import numpy as np

def positional_encoding(pos, d_model):
    def cal_angle(position, i):
        return position / np.power(10000, int(i) / d_model)

    def get_posi_angle_vec(position):
        return [cal_angle(position, i) for i in range(d_model)]

    sinusoid_table = np.array([get_posi_angle_vec(pos_i) for pos_i in range(pos)])

    sinusoid_table[:, 0::2] = np.sin(sinusoid_table[:, 0::2])
    sinusoid_table[:, 1::2] = np.cos(sinusoid_table[:, 1::2])

    return sinusoid_table

pos = 7
d_model = 4
i = 0

print("Positional Encoding 값:\n", positional_encoding(pos, d_model))

print("")
print("if pos == 0, i == 0: ", np.sin(0 / np.power(10000, 2 * i / d_model)))
print("if pos == 1, i == 0: ", np.sin(1 / np.power(10000, 2 * i / d_model)))
print("if pos == 2, i == 0: ", np.sin(2 / np.power(10000, 2 * i / d_model)))
print("if pos == 3, i == 0: ", np.sin(3 / np.power(10000, 2 * i / d_model)))

print("")
print("if pos == 0, i == 1: ", np.cos(0 / np.power(10000, 2 * i + 1 / d_model)))
print("if pos == 1, i == 1: ", np.cos(1 / np.power(10000, 2 * i + 1 / d_model)))
print("if pos == 2, i == 1: ", np.cos(2 / np.power(10000, 2 * i + 1 / d_model)))
print("if pos == 3, i == 1: ", np.cos(3 / np.power(10000, 2 * i + 1 / d_model)))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 7))
plt.imshow(positional_encoding(100, 300), cmap='Blues')
plt.show()

## Scaled Dot-Product Attention

In [ ]:
import torch
import matplotlib.pyplot as plt

def make_dot_product_tensor(shape):
    A = torch.rand(shape) * 6 - 3  # uniform distribution between -3 and 3
    B = (torch.rand(shape) * 6 - 3).transpose(0, 1)
    
    return torch.tensordot(A, B, dims=1)

length = 30
big_dim = 1024.
small_dim = 10.

big_tensor = make_dot_product_tensor((length, int(big_dim)))
scaled_big_tensor = big_tensor / torch.sqrt(torch.tensor(big_dim))
small_tensor = make_dot_product_tensor((length, int(small_dim)))
scaled_small_tensor = small_tensor / torch.sqrt(torch.tensor(small_dim))

fig = plt.figure(figsize=(13, 6))

ax1 = fig.add_subplot(141)
ax2 = fig.add_subplot(142)
ax3 = fig.add_subplot(143)
ax4 = fig.add_subplot(144)

ax1.set_title('1) Big Tensor')
ax2.set_title('2) Big Tensor(Scaled)')
ax3.set_title('3) Small Tensor')
ax4.set_title('4) Small Tensor(Scaled)')

ax1.imshow(torch.softmax(big_tensor, dim=-1).detach().numpy(), cmap='cividis')
ax2.imshow(torch.softmax(scaled_big_tensor, dim=-1).detach().numpy(), cmap='cividis')
ax3.imshow(torch.softmax(small_tensor, dim=-1).detach().numpy(), cmap='cividis')
ax4.imshow(torch.softmax(scaled_small_tensor, dim=-1).detach().numpy(), cmap='cividis')

plt.show()


## 인과 관계 마스킹

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

def make_dot_product_tensor(shape):
    A = torch.rand(shape) * 6 - 3  # uniform distribution between -3 and 3
    B = (torch.rand(shape) * 6 - 3).transpose(0, 1)
    
    return torch.tensordot(A, B, dims=1)

def generate_causality_mask(seq_len):
    mask = 1 - np.cumsum(np.eye(seq_len, seq_len), 0)
    return mask

sample_tensor = make_dot_product_tensor((20, 512))
sample_tensor = sample_tensor / torch.sqrt(torch.tensor(512.))

mask = generate_causality_mask(sample_tensor.shape[0])

fig = plt.figure(figsize=(7, 7))

ax1 = fig.add_subplot(121)
ax2 = fig.add_subplot(122)

ax1.set_title('1) Causality Mask')
ax2.set_title('2) Masked Attention')

ax1.imshow((torch.ones(sample_tensor.shape) + torch.tensor(mask)).detach().numpy(), cmap='cividis')

mask *= -1e9
ax2.imshow(torch.softmax(sample_tensor + torch.tensor(mask), dim=-1).detach().numpy(), cmap='cividis')

plt.show()

## Learning Rate Schedular

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

d_model = 512
warmup_steps = 4000

lrates = []
for step_num in range(1, 50000):
    lrate = (np.power(d_model, -0.5)) * np.min(
        [np.power(step_num, -0.5), step_num * np.power(warmup_steps, -1.5)])
    lrates.append(lrate)

plt.figure(figsize=(6, 3))
plt.plot(lrates)
plt.show()